### 경로 셋팅

In [ ]:
cd /content/sample_data/

## Groq API

Groq API를 활용하여 실습을 진행할 예정입니다.

아래의 링크를 따라 들어가서 직접 API key를 발급 받으시면 됩니다.

1. 웹사이트에 방문해서 회원가입을 진행해 주세요! (https://console.groq.com/home)

2. API key를 발급받은 후 아래의 코드에 key를 복사해서 넣어주세요! (https://console.groq.com/keys)

In [ ]:
!pip install groq
!pip install openai
!pip install datasets
!pip install gym
!pip install requests
!pip install bs4
!pip install langchain-groq
!pip install -U datasets huggingface_hub fsspec

In [ ]:
GROQ_API_KEY=""

In [ ]:
from groq import Groq

client = Groq(
    api_key = GROQ_API_KEY
)

In [ ]:
OPENAI_API_KEY=""

In [ ]:
from openai import OpenAI

client_openai = OpenAI(
    api_key = OPENAI_API_KEY
)

# Chat Completion & LangChain

*Chat Completion?
- LLM을 대화 형식으로 호출하기 위한 interface

*구성요소?
- System Message: 모델에게 역할, 말투, 지침, 행동 제약 등을 설정하기 위한 메시지  
ex. "You are a helpful and concise assistant."

- User Message: 실제 사용자의 질문, 명령, 입력을 담는 메시지  
ex. "파이썬으로 정렬하는 법 알려줘"

- Assistant Message: 모델의 이전 응답을 나타내는 메시지 (chat history 관리 등을 위해 쓰임)  
ex. 모델의 이전 응답을 나타내는 메시지"

In [ ]:
# *예시
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "수학 문제를 도와줘"},
    {"role": "assistant", "content": "물론이죠! 어떤 문제인가요?"}
]

In [ ]:
system_message = "You are a helpful assistant." # 어시스턴트의 행동을 설정하고, 대화 동안 어떻게 행동할 지에 대한 특정 명령어를 제공
human_message = "Hello World!" # 어시스턴트가 응답해야 하는 사용자의 메세지 설정 (보통 저희가 ChatGPT에 작성하는 프롬프트라고 생각하시면 간단합니다)

chat_completion = client.chat.completions.create(
    messages=[
        {
            "role": "system",
            "content": system_message
        },
        {
            "role": "user",
            "content": human_message,
        }
    ],

    # 응답을 생성할 모델을 지정하는 과정
    model="llama3-8b-8192" # 오른쪽에 있는 사이트에 나와 있는 모델들로 변경해 가면서 사용할 수 있습니다. https://console.groq.com/docs/models
)

print(chat_completion)

In [ ]:
system_message = "You are a helpful assistant." # 어시스턴트의 행동을 설정하고, 대화 동안 어떻게 행동할 지에 대한 특정 명령어를 제공
human_message = "Hello World!" # 어시스턴트가 응답해야 하는 사용자의 메세지 설정 (보통 저희가 ChatGPT에 작성하는 프롬프트라고 생각하시면 간단합니다)

chat_completion = client_openai.chat.completions.create(
    messages=[
        {
            "role": "system",
            "content": system_message
        },
        {
            "role": "user",
            "content": human_message,
        }
    ],

    # 응답을 생성할 모델을 지정하는 과정
    model="gpt-4o-mini"
    #model="gpt-3.5-turbo-0125"
)

print(chat_completion)

In [ ]:
# import anthropic

# client = anthropic.Anthropic(api_key="your-anthropic-key")

# response = client.messages.create(
#     messages=[
#         {"role": "user", "content": "What is the capital of France?"}
#     ]
#     model="claude-3-5-sonnet-20241022",
# )

# print(response.content[0].text)

In [ ]:
### TODO: LLM의 응답만 처리해서 출력해 보기
### Hint: chat_completion의 dictionary 형태를 자세히 보면 알 수 있습니다!
print(chat_completion.choices[0].message.content)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq

chat = ChatGroq(temperature=0, model_name="llama3-8b-8192", api_key=GROQ_API_KEY)

system = "You are a helpful assistant."
human = "{text}"
prompt_template = ChatPromptTemplate.from_messages([("system", system), ("human", human)])

chain = prompt_template | chat
response = chain.invoke({"text": "Hello World!."})

In [ ]:
### TODO: LLM의 응답만 처리해서 출력해 보기
### Hint: response의 dictionary 형태를 자세히 보면 알 수 있습니다!
print(response.content)

## 1-1. In-Context Learning

모델의 크기가 커질수록, 모델을 few-shot demonstration에서 배울 수 있는 능력이 생깁니다.

Zero shot < One shot < Few shot

### Sentiment classification w/ ICL (Practice)

In [ ]:
from datasets import load_dataset
import pandas as pd
import random

dataset_test = load_dataset('imdb',split='test')
short_reviews = dataset_test.filter(lambda x: len(x['text']) < 200)
df = pd.DataFrame(short_reviews)
test_df = df.sample(n=30)

In [ ]:
print(f"data length: {len(test_df)}")
test_df.head()

In [ ]:
# check the first data
print(test_df.iloc[0]['text'])

Preprocess

In [ ]:
LABEL2LANG = {
    0: "negative",
    1: "positive"
}
LANG2LABEL = {
    "negative": 0,
    "positive": 1
}
# List of dictionary
data = []

for index, row in test_df.iterrows():
    data.append({
        "idx" : index,
        "text": row['text'],
        "label": row['label'],
        "label_lang": LABEL2LANG[row['label']]
    })

Prompt setting

In [ ]:
prompt_text1 = """
This movie is awesome! -> positive
This movie is terrible! -> negative
{text} ->"""


prompt_text2 = """
[Example 1]
text: This movie is awesome!
label: positive
[Example 2]
text: This movie is terrible!
label: negative
[Example 3]
text: {text}
label:"""

## TODO ##
prompt_text3 = """
[Example 1]
text: This movie is awesome!
label: positive
[Example 2]
text: This movie is terrible!
label: negative
[Example 3]
text: {text}
label:"""

selected_prompt = prompt_text1

In [ ]:
from tqdm import tqdm
from copy import deepcopy

system_message = "You are a helpful assistant."
human_message = "Hello World!"

def groq_chat(prompt, stop_tokens=None):
    """Groq API 호출 함수"""
    chat_completion = client.chat.completions.create(
        model="llama3-8b-8192",
        messages=[
            {
                "role": "system",
                "content": system_message
            },
            {
                "role": "user",
                "content": prompt
            }
        ]
    )
    return chat_completion.choices[0].message.content

def test_prompt(prompt_template, data):
    answer_list = []
    for d in tqdm(data):
        tmp = deepcopy(d)

        input_text = prompt_template.format(text=d['text'])

        response_content = groq_chat(input_text)
        tmp['response'] = response_content

        cleaned_response = response_content.strip().lower()
        tmp['cleaned_response'] = cleaned_response

        try:
            answer = LANG2LABEL[cleaned_response]
        except:
            print(f"\nidx: {d['idx']}, Error: {cleaned_response}")
            answer = -1
        tmp['answer'] = answer
        answer_list.append(tmp)
    return answer_list

def get_accuracy(data):
  correct = 0
  print("calculating accuracy")
  for d in tqdm(data):
      if d['answer'] == d['label']:
          correct += 1
  return correct / len(data)

In [ ]:
answer_list = test_prompt(prompt_template, data)
print(f"\nAccuracy: {get_accuracy(answer_list)}")

### Math reasoning w/ ICL

Loading test dataset (GSM8K) with Hugginface library

In [ ]:
from datasets import load_dataset

gsm8k = load_dataset("gsm8k", "main")['test'] # For testing
gsm8k_train = load_dataset("gsm8k", "main")['train'] # For examples

In [ ]:
print("Question:")
for l in gsm8k['question'][0].split("."):
    print(l)
print("="*100)
print("Answer:")
print(gsm8k['answer'][0])

Building Prompt

In [ ]:
### TODO: n-shot prompt를 만들어 보기!
### Train dataset에서 n 개의 예시를 랜덤으로 뽑아와서 shot으로 넣어보는 과정입니다.
import random

def construct_direct_prompt(num_exemplars: int): # num_exemplars: 사용할 shot의 개수
    # Hint: random 하게 넣어줘야 하니, random하게 indices를 sampling해야 합니다.
    # random.sample(indices_list, num) -> indices_list에서 num 만큼 샘플링하기
    sampled_indices = random.sample([i for i in range(len(gsm8k_train['question']))], num_exemplars)

    # GSM8K에서 few-shot demonstration을 넣은 프롬프트를 작성하시면 됩니다.
    # Hint: 위 셀에 있는 GSM8K 데이터셋의 형태를 잘 보시면 됩니다.
    # Demonstration이 들어가는 형태는 원하는 형태로 바꿔서 사용하시면 됩니다.
    prompt = ""
    tmp = 0
    for i in sampled_indices:
      cur_question = gsm8k_train['question'][i] # 데이터셋에서 예시로 넣을 질문 가져오기
      cur_answer = gsm8k_train['answer'][i].split("####")[-1].strip() # 데이터셋에서 위 질문의 정답 가져오기. 문자열에서 답만 가져오려면 어떻게 해야 할까요?
      prompt += f"[Example {tmp+1}]\n" # Few-shot 형태로 넣어주기. 여러 exemplars을 보여주려면 어떻게 해야할까요?
      prompt += f"Question:\n{cur_question}\n"
      prompt += f"Answer:{cur_answer}\n"
      tmp += 1

    prompt += f"\n[Example {num_exemplars+1}]\n"
    prompt += "Question:\n{question}\nAnswer:"

    return prompt

Example prompt

```
[Example 1]
Question:
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?
Answer:72

[Example 2]
Question:
Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn?
Answer:10

[Example 3]
Question:
Betty is saving money for a new wallet which costs $100. Betty has only half of the money she needs. Her parents decided to give her $15 for that purpose, and her grandparents twice as much as her parents. How much more money does Betty need to buy the wallet?
Answer:5

[Example 4]
Question: {question}
Answer:
```

In [ ]:
direct_prompt_0shot = construct_direct_prompt(0)
direct_prompt_1shot = construct_direct_prompt(1)
direct_prompt_3shot = construct_direct_prompt(3)

Let's check the accuracy of the model!

In [ ]:
prompt = direct_prompt_0shot # direct_prompt_1shot, direct_prompt_3shot
prompt

In [ ]:
from tqdm import tqdm
import json
import re

results_collected = []
pass_collected = []
VERBOSE = False # If you want to check some of the results first
def parse_model_responses(text):
    # Enhancing regex to capture numbers possibly associated with units or other contexts
    regex = r"(?:Answer:|Model response:)\s*\$?([0-9,]+)\b|([0-9,]+)\s*(meters|cups|miles|minutes)"
    matches = re.finditer(regex, text, re.MULTILINE)
    results = [match.group(1) if match.group(1) else match.group(2).replace(",", "") for match in matches]

    # If no matches found in previous patterns, attempt to retrieve simpler numeric or monetary values
    if len(results) == 0:
        results.append(text.split("Answer:")[-1].strip())

    if VERBOSE:
        print(f"Results found: {results}")
    return results[-1] if results else None

for i in tqdm(range(50)):
    cur_question = gsm8k['question'][i]
    cur_answer = gsm8k['answer'][i].split("####")[-1].strip()
    cur_model_input = prompt.format(question=cur_question)
    result = client.chat.completions.create(
        messages=[
            {
                "role": "system",
                "content": "You are a helpful assistant."
            },
            {
                "role": "user",
                "content": cur_model_input
            }
        ],

        # The language model which will generate the completion.
        model="llama3-8b-8192" # You can change the model written in https://console.groq.com/docs/models
    ).choices[0].message.content

    if "Error:" in result:
        print(result)
        break

    cur_prediction = parse_model_responses(result)
    if VERBOSE:
        print("Raw response:", result)
        print("Predicted answer:",cur_prediction)
        print("Reference answer:",cur_answer)

    pass_collected.append(cur_prediction.strip().replace("$", "0") == cur_answer)
    results_collected.append({"question": cur_question, "answer": cur_answer, "prediction": cur_prediction})
    print(f"Acc: {sum(pass_collected)/ len(pass_collected)}")

    if VERBOSE and i == 10:
        break

    with open("result_3shot_direct.json", "w") as f:
        json.dump(results_collected, f, indent=4)

## 1-2. Instruction Tuning

모델에 적절한 명령어를 넣어주게 된다면 성능은 어떻게 변화할까요?

In [ ]:
### TODO: Instruction이 포함된 n-shot prompt를 만들어 보기!
### 위에서 사용했던 코드에 명령어만 추가적으로 넣어주시면 됩니다!
import random

def construct_direct_instruction(num_exemplars):
    sampled_indices = random.sample([i for i in range(len(gsm8k_train['question']))], num_exemplars) # Hint: "gsm8k_train"에서 random 모듈을 활용하여 랜덤하게 인덱스를 sampling하는 코드를 작성하시면 됩니다. 개수는 num_examplars 변수를 활용하시면 됩니다.

    instruction = "Instruction:\nSolve the following mathematical question and generate ONLY the answer after a tag, 'Answer:' without any rationale." # 어떤 명령어를 넣어야 모델이 잘 이해할 지를 고민하시면서 명령어를 넣어주시면 됩니다.

    # GSM8K에서 few-shot demonstration을 넣은 프롬프트를 작성하시면 됩니다.
    # Hint: 위 셀에 있는 GSM8K 데이터셋의 형태를 잘 보시면 됩니다.
    # Demonstration이 들어가는 형태는 원하는 형태로 바꿔서 사용하시면 됩니다.
    prompt = instruction
    tmp = 0
    for i in sampled_indices:
      cur_question = gsm8k_train['question'][i] # 데이터셋에서 예시로 넣을 질문 가져오기
      cur_answer = gsm8k_train['answer'][i].split("####")[-1].strip() # 데이터셋에서 위 질문의 정답 가져오기
      prompt += f"\n[Example {tmp+1}]\n" # Few-shot 형태로 넣어주기
      prompt += f"Question:\n{cur_question}\n"
      prompt += f"Answer:{cur_answer}\n"
      tmp += 1

    prompt += f"\n[Example {num_exemplars+1}]\n"
    prompt += "Question:\n{question}\nAnswer:"

    return prompt

Example prompt

```
Instruction:
Solve the following mathematical question and generate ONLY the answer after a tag, 'Answer:' without any rationale.
[Example 1]
Question:
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?
Answer:72

[Example 2]
Question:
Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn?
Answer:10

[Example 3]
Question:
Betty is saving money for a new wallet which costs $100. Betty has only half of the money she needs. Her parents decided to give her $15 for that purpose, and her grandparents twice as much as her parents. How much more money does Betty need to buy the wallet?
Answer:5

[Example 4]
Question: {question}
Answer:
```

In [ ]:
direct_instruction_0shot = construct_direct_instruction(0)
direct_instruction_1shot = construct_direct_instruction(1)
direct_instruction_3shot = construct_direct_instruction(3)

모델이 얼마나 잘 맞추는 지를 확인해 보겠습니다.

In [ ]:
prompt = direct_instruction_0shot # direct_instruction_1shot, direct_instruction_3shot
prompt

In [ ]:
from tqdm import tqdm
import json
import re

results_collected = []
pass_collected = []
VERBOSE = False # 일부 결과만 미리 확인하고 싶을 때!

for i in tqdm(range(50)):
    cur_question = gsm8k['question'][i]
    cur_answer = gsm8k['answer'][i].split("####")[-1].strip()
    cur_model_input = prompt.format(question=cur_question)
    result = client.chat.completions.create(
        messages=[
            {
                "role": "system",
                "content": "You are a helpful assistant."
            },
            {
                "role": "user",
                "content": cur_model_input
            }
        ],

        model="llama3-8b-8192"
    ).choices[0].message.content  ### TODO: 위에서 확인했던 chat_completion 모듈을 활용해서 모델의 응답을 얻어보겠습니다!

    if "Error:" in result:
        print(result)
        break

    cur_prediction = parse_model_responses(result)
    if VERBOSE:
        print("Raw response:", result)
        print("Predicted answer:",cur_prediction)
        print("Reference answer:",cur_answer)

    pass_collected.append(cur_prediction.strip().replace("$", "0") == cur_answer)
    results_collected.append({"question": cur_question, "answer": cur_answer, "prediction": cur_prediction})
    print(f"Acc: {sum(pass_collected)/ len(pass_collected)}")

    if VERBOSE and i == 10:
        break

    ### 저장하고 싶은 형태의 이름으로 변경하여서 저장하여도 무방합니다!
    with open("result_3shot_instruction.json", "w") as f:
        json.dump(results_collected, f, indent=4)

# 2-1. Chain of Thoughts Prompting

## Few shot CoT

Generate intermediate reasoning steps by providing few-shot demonstrations

In [ ]:
def construct_cot_prompt(num_exemplars):
  sampled_indices = random.sample([i for i in range(len(gsm8k_train['question']))], num_exemplars)

  # CoT의 few-shot demonstration을 한번 만들어 보겠습니다!
  prompt = ""
  tmp = 0
  for i in sampled_indices:
    cur_question = gsm8k_train['question'][i] # 데이터셋에서 예시로 넣을 질문 가져오기
    cur_answer = gsm8k_train['answer'][i] # 데이터셋에서 위 질문의 정답 가져오기
    prompt += f"[Example {tmp+1}]\n" # Few-shot 형태로 넣어주기
    prompt += f"Question:\n{cur_question}\n"
    prompt += f"Answer:{cur_answer}\n"
    tmp += 1
    # TODO: Few-shot demonstration으로 추론 과정을 넣어주세요. 위의 코드 및 아래 예시를 참고하면서 진행하시면 됩니다.

  prompt += f"\n[Example {num_exemplars+1}]\n"
  prompt += "Question:\n{question}\n"
  prompt += "Output:"

  return prompt

def construct_cot_instruction(num_exemplars):
  sampled_indices = random.sample([i for i in range(len(gsm8k_train['question']))], num_exemplars)

  instruction = "Instruction:\nGenerate the answer for the given mathematical question with step-by-step rationale toward the answer. Provide your final answer based on the rationale using an identifier '####'." # TODO

  # Constructing a prompt with few-shot demonstrations from GSM8K
  prompt = instruction
  tmp = 0
  for i in sampled_indices:
    cur_question = gsm8k_train['question'][i] # 데이터셋에서 예시로 넣을 질문 가져오기
    cur_answer = gsm8k_train['answer'][i].split("####")[-1].strip() # 데이터셋에서 위 질문의 정답 가져오기
    prompt += f"\n[Example {tmp+1}]\n" # Few-shot 형태로 넣어주기
    prompt += f"Question:\n{cur_question}\n"
    prompt += f"Answer:{cur_answer}\n"
    tmp += 1
      # TODO: Few-shot demonstration으로 추론 과정을 넣어주세요. 위의 코드 및 아래 예시를 참고하면서 진행하시면 됩니다.
  prompt += f"\n[Example {num_exemplars+1}]\n"
  prompt += "Question:\n{question}\n"
  prompt += "Output:"

  return prompt

Example prompt

```
Instruction:
Generate the answer for the given mathematical question with step-by-step rationale toward the answer. Provide your final answer based on the rationale using an identifier '####'.
[Example 1]
Question:
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?
Output:Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72

[Example 2]
Question:
Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn?
Output:Weng earns 12/60 = $<<12/60=0.2>>0.2 per minute.
Working 50 minutes, she earned 0.2 x 50 = $<<0.2*50=10>>10.
#### 10

[Example 3]
Question:
Betty is saving money for a new wallet which costs $100. Betty has only half of the money she needs. Her parents decided to give her $15 for that purpose, and her grandparents twice as much as her parents. How much more money does Betty need to buy the wallet?
Output:In the beginning, Betty has only 100 / 2 = $<<100/2=50>>50.
Betty's grandparents gave her 15 * 2 = $<<15*2=30>>30.
This means, Betty needs 100 - 50 - 30 - 15 = $<<100-50-30-15=5>>5 more.
#### 5

[Example 4]
Question: {question}
Output:
```

In [ ]:
cot_prompt_1shot = construct_cot_prompt(1)
cot_prompt_3shot = construct_cot_prompt(3)

cot_instruction_3shot = construct_cot_instruction(3)

In [ ]:
prompt = cot_prompt_3shot # cot_prompt_1shot / cot_instruction_3shot
prompt

Let's check the accuracy of the model!

In [ ]:
VERBOSE=False # If you want to check some of the results first
results_collected = []
pass_collected = []

def parse_model_responses(text):
    # Enhancing regex to capture numbers possibly associated with units or other contexts
    regex = r"(?:Answer:|Model response:)\s*\$?([0-9,]+)\b|([0-9,]+)\s*(meters|cups|miles|minutes)"
    matches = re.finditer(regex, text, re.MULTILINE)
    results = [match.group(1) if match.group(1) else match.group(2).replace(",", "") for match in matches]

    # If no matches found in previous patterns, attempt to retrieve simpler numeric or monetary values
    if len(results) == 0:
        results.append(text.split("####")[-1].strip())

    if VERBOSE:
        print(f"Results found: {results}")
    return results[-1] if results else None

for i in tqdm(range(50)):
    cur_question = gsm8k['question'][i]
    cur_answer = gsm8k['answer'][i].split("####")[-1].strip()
    cur_model_input = prompt.format(question=cur_question)
    result = client.chat.completions.create(
        messages=[
            {
                "role": "system",
                "content": "You are a helpful assistant."
            },
            {
                "role": "user",
                "content": cur_model_input
            }
        ],


        model="llama3-8b-8192"
    ).choices[0].message.content

    if "Error:" in result:
        print(result)
        break

    cur_prediction = parse_model_responses(result)
    if VERBOSE:
        print("Raw response:", result)
        print("Predicted answer:",cur_prediction)
        print("Reference answer:",cur_answer)

    pass_collected.append(cur_prediction.strip().replace("$", "0") == cur_answer)
    results_collected.append({"question": cur_question, "answer": cur_answer, "prediction": cur_prediction})
    print(f"Acc: {sum(pass_collected)/ len(pass_collected)}")

    if VERBOSE and i == 10:
        break

    with open("result_3shot_cot.json", "w") as f:
        json.dump(results_collected, f, indent=4)

## Zero shot CoT



In [ ]:
def construct_0shot_cot():
  prompt = ""

  prompt += "Question:\n{question}\n"
  prompt += "Output:"

  return prompt

어떻게 추론 과정과 정답을 추출해 낼 수 있을까요?

In [ ]:
prompt = construct_0shot_cot()

In [ ]:
VERBOSE=False # If you want to check some of the results first
results_collected = []
pass_collected = []

for i in tqdm(range(50)):
    cur_question = gsm8k['question'][i]
    cur_answer = gsm8k['answer'][i].split("####")[-1].strip()
    cur_model_input = prompt.format(question=cur_question)
    result = client.chat.completions.create(
    messages=[
        {
            "role": "system",
            "content": "You are a helpful assistant."
        },
        {
            "role": "user",
            "content": cur_model_input
        },
        {
            "role": "assistant",
            "content": "Let's think step by step"
        }
    ],

    model="llama3-8b-8192"
  ).choices[0].message.content ### Zero-shot CoT를 위한 응답 생성 chat completion을 작성해 보겠습니다! 어떻게 completion 작성해야 할 지 고민해 보시면서 작성해 주세요.
    ### Hint: LLM이 답변을 내놓을 때에는 "role": "assistant"를 활용하여 작성합니다.

    if "Error:" in result:
        print(result)
        break

    cur_prediction = parse_model_responses(result)
    if VERBOSE:
        print("Raw response:", result)
        print("Predicted answer:",cur_prediction)
        print("Reference answer:",cur_answer)

    pass_collected.append(cur_prediction.strip().replace("$", "0") == cur_answer)
    results_collected.append({"question": cur_question, "answer": cur_answer, "prediction": cur_prediction})
    print(f"Acc: {sum(pass_collected)/ len(pass_collected)}")

    if VERBOSE and i == 10:
        break

    with open("result_0shot_cot.json", "w") as f:
        json.dump(results_collected, f, indent=4)

명령어를 가지고 만들어 내는 Zero-shot CoT와 비교해보면??

In [ ]:
prompt = construct_cot_instruction(0)
prompt

In [ ]:
VERBOSE=False # If you want to check some of the results first
results_collected = []
pass_collected = []

for i in tqdm(range(50)):
    cur_question = gsm8k['question'][i]
    cur_answer = gsm8k['answer'][i].split("####")[-1].strip()
    cur_model_input = prompt.format(question=cur_question)
    result = result = client.chat.completions.create(
        messages=[
            {
                "role": "system",
                "content": "You are a helpful assistant."
            },
            {
                "role": "user",
                "content": cur_model_input
            }
        ],


        model="llama3-8b-8192"
    ).choices[0].message.content ### TODO

    if "Error:" in result:
        print(result)
        break

    cur_prediction = parse_model_responses(result)
    if VERBOSE:
        print("Raw response:", result)
        print("Predicted answer:",cur_prediction)
        print("Reference answer:",cur_answer)

    pass_collected.append(cur_prediction.strip().replace("$", "0") == cur_answer)
    results_collected.append({"question": cur_question, "answer": cur_answer, "prediction": cur_prediction})
    print(f"Acc: {sum(pass_collected)/ len(pass_collected)}")

    if VERBOSE and i == 10:
        break

    with open("result_cot_0shot.json", "w") as f:
        json.dump(results_collected, f, indent=4)

# 2-2. Self-consistency
답변을 여러 번 생성하면 성능이 더 좋아질까요?

In [ ]:
VERBOSE=False
results_collected = []
pass_collected = []

for i in tqdm(range(50)):
    cur_question = gsm8k['question'][i]
    cur_answer = gsm8k['answer'][i].split("####")[-1].strip()
    answer_dict = {}

    ### TODO: 위의 코드들을 참고하여, 총 5번 응답을 생성하고, 가장 많이 나온 응답을 정답으로 선택하세요! 만약 전부 다 다른 응답이 나온다면, 처음 응답을 정답으로 선택해 주세요.
    for j in range(5):
        prompt = construct_cot_prompt(3)
        cur_model_input = prompt.format(question=cur_question)
        result = client.chat.completions.create(
            messages=[
                {
                    "role": "system",
                    "content": "You are a helpful assistant."
                },
                {
                    "role": "user",
                    "content": cur_model_input
                }
            ],


            model="llama3-8b-8192"
        ).choices[0].message.content

        if "Error:" in result:
            print(result)
            break

        cur_response = parse_model_responses(result)
        if cur_response in answer_dict:
            answer_dict[cur_response] += 1
        else:
            answer_dict[cur_response] = 1


    cur_prediction = max(answer_dict, key=answer_dict.get)

    if VERBOSE:
        print("Raw response:", result)
        print("Predicted answer:",cur_prediction)
        print("Reference answer:",cur_answer)

    pass_collected.append(cur_prediction.strip().replace("$", "0") == cur_answer)
    results_collected.append({"question": cur_question, "answer": cur_answer, "prediction": cur_prediction})
    print(f"Acc: {sum(pass_collected)/ len(pass_collected)}")

    if VERBOSE and i == 10:
        break

    ### 저장하고 싶은 형태의 이름으로 변경하여서 저장하여도 무방합니다!
    with open("result_3shot_cot.json", "w") as f:
        json.dump(results_collected, f, indent=4)

# 2-3. ReAct

In [ ]:
def llm(prompt, stop=["\n"]):
    response = client.chat.completions.create(
      messages=[
          {
              "role": "system",
              "content": "You are a helpful assistant."
          },
          {
              "role": "user",
              "content": prompt
          }
      ], ### TODO
      model="llama3-70b-8192", # 오른쪽에 있는 사이트에 나와 있는 모델들로 변경해 가면서 사용할 수 있습니다. https://console.groq.com/docs/models
      temperature=0,
      max_tokens=100,
      top_p=1,
      frequency_penalty=0.0,
      presence_penalty=0.0,
      stop=stop
    ).choices[0].message.content
    return response

### Env setting

아래 코드는 무조건 실행을 해야 하는 코드이긴 하지만, 환경을 위한 코드들이기 때문에, 따로 수정 할 필요 없이 실행만 시키시면 됩니다!

In [ ]:
import json
import os
import gym
import numpy as np
import re
import string
from collections import Counter


def normalize_answer(s):
  def remove_articles(text):
    return re.sub(r"\b(a|an|the)\b", " ", text)

  def white_space_fix(text):
      return " ".join(text.split())

  def remove_punc(text):
      exclude = set(string.punctuation)
      return "".join(ch for ch in text if ch not in exclude)

  def lower(text):
      return text.lower()

  return white_space_fix(remove_articles(remove_punc(lower(s))))

def f1_score(prediction, ground_truth):
  normalized_prediction = normalize_answer(prediction)
  normalized_ground_truth = normalize_answer(ground_truth)

  ZERO_METRIC = (0, 0, 0)

  if normalized_prediction in ['yes', 'no', 'noanswer'] and normalized_prediction != normalized_ground_truth:
    return ZERO_METRIC
  if normalized_ground_truth in ['yes', 'no', 'noanswer'] and normalized_prediction != normalized_ground_truth:
    return ZERO_METRIC

  prediction_tokens = normalized_prediction.split()
  ground_truth_tokens = normalized_ground_truth.split()
  common = Counter(prediction_tokens) & Counter(ground_truth_tokens)
  num_same = sum(common.values())
  if num_same == 0:
    return ZERO_METRIC
  precision = 1.0 * num_same / len(prediction_tokens)
  recall = 1.0 * num_same / len(ground_truth_tokens)
  f1 = (2 * precision * recall) / (precision + recall)
  return f1, precision, recall

class HotPotQAWrapper(gym.Wrapper):
  def __init__(self, env, split):
    super().__init__(env)
    data_file = load_dataset("hbhhyj/hotpotqa_dev")["validation"]
    self.data = data_file.to_list()
    self.data = [(d['question'], d['answer']) for d in self.data]
    self.data_idx = 0
    self.split = split

  def reset(self, seed=None, return_info=False, options=None, idx=None):
    self.env.reset(seed=seed, return_info=return_info, options=options)
    try:
      self.env.step('')
    except:
      pass
    self.env.reset(seed=seed, return_info=return_info, options=options)
    self.data_idx = int(np.random.randint(len(self.data))) if idx is None else idx
    observation = f"Question: {self.data[self.data_idx][0]}"
    info = self._get_info()
    return (observation, info) if return_info else observation

  def _get_info(self):
    return {
      "steps": self.steps,
      "answer": self.answer,
      "question": self.data[self.data_idx][0],
      "hotpot_split": self.split
    }

  def get_reward(self, info):
    if info['answer'] is not None:
      pred = normalize_answer(self.data[self.data_idx][1])
      gt = normalize_answer(info['answer'])
      score = (pred == gt)
      return int(score)
    return 0

  def get_metrics(self, info):
    if info['answer'] is not None:
      pred = normalize_answer(self.data[self.data_idx][1])
      gt = normalize_answer(info['answer'])
      em = (pred == gt)
      f1 = f1_score(pred, gt)[0]
      return {'reward': em, 'em': em, 'f1': f1}
    return {'reward': 0, 'em': 0, 'f1': 0}

  def step(self, action):
    # TODO: first step obs does not have question.
    obs, _, done, info = self.env.step(action)
    reward = self.get_reward(info)
    if done:
      obs = f"Episode finished, reward = {reward}\n"
      info.update({"gt_answer": self.data[self.data_idx][1], "question_idx": self.data_idx})
      info.update(self.get_metrics(info))
    return obs, reward, done, info

  def __len__(self):
    return len(self.data)


class LoggingWrapper(gym.Wrapper):
  def __init__(self, env, folder="trajs", file_id=None):
    super().__init__(env)
    self.trajs = []
    self.traj = {"observations": [], "actions": []}
    self.folder = folder
    self.file_id = np.random.randint(0, 10000000) if file_id is None else file_id
    self.file_path = f"{self.folder}/{self.file_id}.json"
    os.makedirs("trajs", exist_ok=True)

  def __len__(self):
    return len(self.env.data)


  def reset(self, seed=None, return_info=False, options=None, idx=None):
    output = self.env.reset(seed=seed, return_info=return_info, options=options, idx=idx)
    observation = output[0] if return_info else output
    self.traj = {"observations": [observation], "actions": []}
    return output

  def step(self, action):
    obs, reward, done, info = self.env.step(action)
    self.traj["observations"].append(obs)
    self.traj["actions"].append(action)
    if done:
      self.traj.update(info)
    return obs, reward, done, info

  def update_record(self):
    if len(self.traj) > 0:
      self.trajs.append(self.traj)
      self.traj = {"observations": [], "actions": []}

  def write(self):
    self.update_record()
    with open(self.file_path, "w") as f:
      json.dump(self.trajs, f)
      print(f"Saved trajs to trajs/{self.file_id}.json")

  def close(self):
    self.write()

In [ ]:
import ast
import json
import time
import gym
import requests
from bs4 import BeautifulSoup

# import wikipedia

def clean_str(p):
  return p.encode().decode("unicode-escape").encode("latin1").decode("utf-8")


class textSpace(gym.spaces.Space):
  def contains(self, x) -> bool:
    """Return boolean specifying if x is a valid member of this space."""
    return isinstance(x, str)


class WikiEnv(gym.Env):

  def __init__(self):
    """
      Initialize the environment.
    """
    super().__init__()
    self.page = None  # current Wikipedia page
    self.obs = None  # current observation
    self.lookup_keyword = None  # current lookup keyword
    self.lookup_list = None  # list of paragraphs containing current lookup keyword
    self.lookup_cnt = None  # current lookup index
    self.steps = 0  # current number of steps
    self.answer = None  # current answer from the agent
    self.observation_space = self.action_space = textSpace()
    self.search_time = 0
    self.num_searches = 0

  def _get_obs(self):
    return self.obs

  def _get_info(self):
    return {"steps": self.steps, "answer": self.answer}

  def reset(self, seed=None, return_info=False, options=None):
    # We need the following line to seed self.np_random
    # super().reset(seed=seed)
    self.obs = ("Interact with Wikipedia using search[], lookup[], and "
                "finish[].\n")
    self.page = None
    self.lookup_keyword = None
    self.lookup_list = None
    self.lookup_cnt = None
    self.steps = 0
    self.answer = None
    observation = self._get_obs()
    info = self._get_info()
    return (observation, info) if return_info else observation

  def construct_lookup_list(self, keyword):
    # find all paragraphs
    if self.page is None:
      return []
    paragraphs = self.page.split("\n")
    paragraphs = [p.strip() for p in paragraphs if p.strip()]

    # find all sentence
    sentences = []
    for p in paragraphs:
      sentences += p.split('. ')
    sentences = [s.strip() + '.' for s in sentences if s.strip()]

    parts = sentences
    parts = [p for p in parts if keyword.lower() in p.lower()]
    return parts

  @staticmethod
  def get_page_obs(page):
    # find all paragraphs
    paragraphs = page.split("\n")
    paragraphs = [p.strip() for p in paragraphs if p.strip()]

    # find all sentence
    sentences = []
    for p in paragraphs:
      sentences += p.split('. ')
    sentences = [s.strip() + '.' for s in sentences if s.strip()]
    return ' '.join(sentences[:5])

    # ps = page.split("\n")
    # ret = ps[0]
    # for i in range(1, len(ps)):
    #   if len((ret + ps[i]).split(" ")) <= 50:
    #     ret += ps[i]
    #   else:
    #     break
    # return ret

  def search_step(self, entity):
    entity_ = entity.replace(" ", "+")
    search_url = f"https://en.wikipedia.org/w/index.php?search={entity_}"
    old_time = time.time()
    response_text = requests.get(search_url).text
    self.search_time += time.time() - old_time
    self.num_searches += 1
    soup = BeautifulSoup(response_text, features="html.parser")
    result_divs = soup.find_all("div", {"class": "mw-search-result-heading"})
    if result_divs:  # mismatch
      self.result_titles = [clean_str(div.get_text().strip()) for div in result_divs]
      self.obs = f"Could not find {entity}. Similar: {self.result_titles[:5]}."
    else:
      page = [p.get_text().strip() for p in soup.find_all("p") + soup.find_all("ul")]
      if any("may refer to:" in p for p in page):
        self.search_step("[" + entity + "]")
      else:
        self.page = ""
        for p in page:
          if len(p.split(" ")) > 2:
            self.page += clean_str(p)
            if not p.endswith("\n"):
              self.page += "\n"
        self.obs = self.get_page_obs(self.page)
        self.lookup_keyword = self.lookup_list = self.lookup_cnt = None

  def step(self, action):
    reward = 0
    done = False
    action = action.strip()
    if self.answer is not None:  # already finished
      done = True
      return self.obs, reward, done, self._get_info()

    if action.startswith("search[") and action.endswith("]"):
      entity = action[len("search["):-1]
      # entity_ = entity.replace(" ", "_")
      # search_url = f"https://en.wikipedia.org/wiki/{entity_}"
      self.search_step(entity)
    elif action.startswith("lookup[") and action.endswith("]"):
      keyword = action[len("lookup["):-1]
      if self.lookup_keyword != keyword:  # reset lookup
        self.lookup_keyword = keyword
        self.lookup_list = self.construct_lookup_list(keyword)
        self.lookup_cnt = 0
      if self.lookup_cnt >= len(self.lookup_list):
        self.obs = "No more results.\n"
      else:
        self.obs = f"(Result {self.lookup_cnt + 1} / {len(self.lookup_list)}) " + self.lookup_list[self.lookup_cnt]
        self.lookup_cnt += 1
    elif action.startswith("finish[") and action.endswith("]"):
      answer = action[len("finish["):-1]
      self.answer = answer
      done = True
      self.obs = f"Episode finished, reward = {reward}\n"
    elif action.startswith("think[") and action.endswith("]"):
      self.obs = "Nice thought."
    else:
      self.obs = "Invalid action: {}".format(action)

    self.steps += 1

    return self.obs, reward, done, self._get_info()

  def get_time_info(self):
    speed = self.search_time / self.num_searches if self.num_searches else 0
    return {
        "call_speed": speed,
        "call_time": self.search_time,
        "num_calls": self.num_searches,
    }


### Let's run ReAct

In [ ]:
env = WikiEnv()
env = HotPotQAWrapper(env, split="dev")
env = LoggingWrapper(env)

def step(env, action):
    attempts = 0
    while attempts < 10:
        try:
            return env.step(action)
        except requests.exceptions.Timeout:
            attempts += 1

In [ ]:
prompt_dict = load_dataset("hbhhyj/react_prompt")["train"]
prompt_dict = prompt_dict.to_list()[0]

prompt_dict

#### Reason only (CoT)

In [ ]:
cotqa_examples = prompt_dict['cotqa_simple6']
### TODO: 위에 있는 prompt를 사용하셔도 되지만, 직접 prompt를 작성해 보시는 것을 추천드립니다. 작성하실 때에는 CoT만 하도록 prompt를 작성해 주시면 됩니다.
instruction = """Generate the answer for the given question answering task with step-by-step rationale toward the answer.
Here are some examples
"""
cotqa_prompt = instruction + cotqa_examples

def cotqa(idx=None, prompt=cotqa_prompt, to_print=True):
    question = env.reset(idx=idx)
    if to_print:
        print(idx, question)
    prompt += question + "\n"
    action = llm(prompt + f" Provide your final answer based on the rationale using an identifier '####'", stop=None)
    action = action.split("####")[-1].strip() ### TODO: Parse the output into the answer.
    action = "finish[" + action + "]"
    obs, r, done, info = step(env, action)

    if to_print:
        print(info, '\n')
    info.update({'n_calls': 1, 'n_badcalls': 0, 'traj': prompt})
    return r, info

In [ ]:
import random
import time
idxs = list(range(7405))
random.Random(0).shuffle(idxs)

rs = []
infos = []
old_time = time.time()
for i in idxs[:20]:
    r, info = cotqa(i, to_print=True)
    # info['em'] -> exact match인지 아닌지 (1 or 0)
    rs.append(info['em'])
    infos.append(info)
    # 맞춘수 총수 정확도 평균시간/문제
    print(sum(rs), len(rs), sum(rs) / len(rs), (time.time() - old_time) / len(rs))
    print('-----------')
    print()

#### Act only

In [ ]:
webtact_examples = prompt_dict['webact_simple6']
### TODO: 위에 있는 prompt를 사용하셔도 되지만, 직접 prompt를 작성해 보시는 것을 추천드립니다. 작성하실 때에는 act와 observe만 할 수 있게 하시면 됩니다.
instruction = """Solve a question answering task with interleaving Action, Observation steps. Action can be three types:
(1) Search[entity], which searches the exact entity on Wikipedia and returns the first paragraph if it exists. If not, it will return some similar entities to search.
(2) Lookup[keyword], which returns the next sentence containing keyword in the current passage.
(3) Finish[answer], which returns the answer and finishes the task.
Here are some examples.
"""
webact_prompt = instruction + webtact_examples

def webact(idx=None, prompt=webact_prompt, to_print=True):
    question = env.reset(idx=idx)
    if to_print:
        print(idx, question)
    prompt += question + "\n"
    n_calls, n_badcalls = 0, 0
    for i in range(1, 8):
        print("i: ", i)
        n_calls += 1
        while True:
            action = llm(prompt + f"Output the Action of this step. You must follow the following format:\nAction {i}:", stop=[f"\nObservation {i}:"])
            try:
                action = action.split(f"Action {i}:")[-1].strip()
                break
            except:
                print('ohh... Let me do it again...')
                n_badcalls += 1
                n_calls += 1
                if n_badcalls == 5:
                    break
        obs, r, done, info = step(env, action[0].lower() + action[1:])
        obs = obs.replace('\\n', '')
        if i == 1:
            prompt += "These are Actions, and Observations of previous steps:\n"
        step_str = f"Action {i}: {action}\nObservation {i}: {obs}\n"
        prompt += step_str
        if to_print:
            print(step_str)
        if done:
            break
    if not done:
        obs, r, done, info = step(env, "finish[]")
    if to_print:
        print(info, '\n')
    info.update({'n_calls': n_calls, 'n_badcalls': n_badcalls, 'traj': prompt})
    return r, info

In [ ]:
import random
import time
idxs = list(range(7405))
random.Random(0).shuffle(idxs)

rs = []
infos = []
old_time = time.time()
for i in idxs[:20]:
    r, info = webact(i, to_print=True)
    rs.append(info['em'])
    infos.append(info)
    print(sum(rs), len(rs), sum(rs) / len(rs), (time.time() - old_time) / len(rs))
    print('-----------')
    print()

#### ReAct

In [ ]:
webthink_examples = prompt_dict['webthink_simple6']
### TODO: 위에 있는 prompt를 사용하셔도 되지만, 직접 prompt를 작성해 보시는 것을 추천드립니다. 작성하실 때에는 추론 후에 act와 observe를 할 수 있게 하시면 됩니다.
instruction = """Solve a question answering task with interleaving Thought, Action, Observation steps. Thought can reason about the current situation, and Action can be three types:
(1) Search[entity], which searches the exact entity on Wikipedia and returns the first paragraph if it exists. If not, it will return some similar entities to search.
(2) Lookup[keyword], which returns the next sentence containing keyword in the current passage.
(3) Finish[answer], which returns the answer and finishes the task.
Here are some examples.
"""
webthink_prompt = instruction + webthink_examples

def webthink(idx=None, prompt=webthink_prompt, to_print=True):
    question = env.reset(idx=idx)
    if to_print:
        print(idx, question)
    prompt += question + "\n"
    n_calls, n_badcalls = 0, 0
    for i in range(1, 8):
        print("i: ", i)
        n_calls += 1
        while True:
            thought_action = llm(prompt + f"Output the Thought and the Action of this step. You must follow the following format:\nThought {i}:\nAction {i}:", stop=[f"\nObservation {i}:"])
            try:
                thought, action = thought_action.strip().split(f"\nAction {i}: ")
                thought = thought.split(f"Thought {i}:")[-1].strip()
                break
            except:
                print('ohh... Let me do it again...')
                print(thought_action)
                n_badcalls += 1
                n_calls += 1
                if n_badcalls == 5:
                    break
        obs, r, done, info = step(env, action[0].lower() + action[1:])
        obs = obs.replace('\\n', '')
        if i == 1:
            prompt += "These are Thoughts, Actions, and Observations of previous steps:\n"
        step_str = f"Thought {i}: {thought}\nAction {i}: {action}\nObservation {i}: {obs}\n"
        prompt += step_str
        if to_print:
            print(step_str)
        if done:
            break
    if not done:
        obs, r, done, info = step(env, "finish[]")
    if to_print:
        print(info, '\n')
    info.update({'n_calls': n_calls, 'n_badcalls': n_badcalls, 'traj': prompt})
    return r, info

In [ ]:
import random
import time
idxs = list(range(7405))
random.Random(0).shuffle(idxs)

rs = []
infos = []
old_time = time.time()
for i in idxs[:20]:
    r, info = webthink(i, to_print=True)
    rs.append(info['em'])
    infos.append(info)
    print(sum(rs), len(rs), sum(rs) / len(rs), (time.time() - old_time) / len(rs))
    print('-----------')
    print()